# Προεπεξεργασία δεδομένων

Από την εξαγωγή (`export_gps_v1.csv`) στο τελικό σύνολο δεδομένων (`final_dataset_thess9.csv`), που είναι η είσοδος του `train_eval_fold.ipynb` και της μεθόδου Barnett & Onnela.

**Βήματα:** sessions → σημεία → γεωγραφικά όρια → σημεία με Δt = 0 → κεραίες → μετατοπίσεις → σημαίες B → ταχύτητα → αλλαγή θερινής ώρας → διαδρομές εκτός Θεσσαλονίκης.

Σε κάθε βήμα οι μετατοπίσεις `dt` (ώρες), `dx`, `dy` (m) είναι εμπρόσθιες διαφορές, από το τρέχον προς το επόμενο σημείο. Οι συντεταγμένες `x, y` είναι στο ΕΓΣΑ87 (EPSG:2100).


In [ ]:
import ast, io
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import numpy as np, pandas as pd
from pyproj import Transformer

INPUT_CSV = "export_gps_v1.csv"
OUTPUT_CSV = "final_dataset_thess9.csv"
counts = {}                                 


def forward_diffs(df, decimals=(6, 2)):
    """dt (ώρες), dx, dy ως το επόμενο σημείο και αθροιστικός χρόνος t_j. Στο τελευταίο σημείο 0."""
    t = pd.to_datetime(df.real_time)
    df["dt"] = ((t.shift(-1) - t).dt.total_seconds() / 3600.0).round(decimals[0])
    df["dx"] = (df.x.shift(-1) - df.x).round(decimals[1])
    df["dy"] = (df.y.shift(-1) - df.y).round(decimals[1])
    df["t_j"] = df.dt.shift(1).fillna(0.0).cumsum().round(5)
    df.loc[df.index[-1], ["dx", "dy", "dt"]] = [0.0, 0.0, 0.0]
    return df


def csv_roundtrip(df):
    """Εγγραφή και ανάγνωση CSV"""
    return pd.read_csv(io.StringIO(df.to_csv(index=False)))


def reindex(df):
    """Συνεχής αρίθμηση μετά από διαγραφή σημείων."""
    df = df.reset_index(drop=True)
    df["point_idx"] = df.index
    return df

## 1. Από τα sessions στα σημεία

Κάθε γραμμή της εξαγωγής είναι ένα session με λίστες πλάτους, μήκους και χρόνου (ms). Αφαιρούνται τα διπλότυπα sessions (ίδια `timestamp`, `sessionID`, `nofPoints`) και κάθε session διασπάται σε σημεία.

- Μέσα στο session: B = 1 και μετατοπίσεις ως το επόμενο σημείο.
- Τελευταίο σημείο του session: B = 0, `dt` ως το πρώτο σημείο του επόμενου session και `dx = dy = 0`.
- Ώρα σε τοπική ώρα Ελλάδας.

In [ ]:
raw = pd.read_csv(INPUT_CSV)
n0 = len(raw)
raw = raw.drop_duplicates(subset=["timestamp", "sessionID", "nofPoints"], keep="first").reset_index(drop=True)
print(f"διπλότυπα sessions: {n0 - len(raw)} | sessions: {len(raw)}")

to_egsa = Transformer.from_crs("epsg:4326", "epsg:2100", always_xy=True)
athens = ZoneInfo("Europe/Athens")


def parse(row):
    try:
        return (ast.literal_eval(str(row.latitude)), ast.literal_eval(str(row.longitude)),
                ast.literal_eval(str(row.elapsedRealTimeNanos)))
    except (ValueError, SyntaxError):
        return None


sessions = [parse(r) for r in raw.itertuples()]
rows, t_hours = [], 0.0
for k, s in enumerate(sessions):
    if s is None:
        continue
    lats, lons, ts = s
    xs, ys = to_egsa.transform(lons, lats)
    for i in range(len(lats)):
        x, y = round(xs[i], 2), round(ys[i], 2)
        if i < len(lats) - 1:                                  # μέσα στο session
            dt = (ts[i + 1] - ts[i]) / 3.6e6
            dx, dy, b = round(round(xs[i + 1], 2) - x, 2), round(round(ys[i + 1], 2) - y, 2), 1
        elif k < len(raw) - 1:                                 # τέλος session: χρόνος ως το επόμενο
            nxt = sessions[k + 1]
            dt = (nxt[2][0] - ts[i]) / 3.6e6 if nxt is not None and len(nxt[2]) else 0.0
            dx, dy, b = 0.0, 0.0, 0
        else:                                                  # τέλος αρχείου
            dt, dx, dy, b = 0.0, 0.0, 0.0, 0
        rows.append({"point_idx": len(rows),
                     "real_time": datetime.fromtimestamp(ts[i] / 1000.0, tz=timezone.utc).astimezone(athens)
                                          .strftime("%Y-%m-%d %H:%M:%S"),
                     "t_j": round(t_hours, 5), "lat": lats[i], "lon": lons[i],
                     "dt": round(dt, 6), "dx": dx, "dy": dy, "x": x, "y": y, "B": b})
        t_hours += dt
df = pd.DataFrame(rows)
counts["Αφαίρεση διπλότυπων sessions"] = len(df)
print(f"σημεία: {len(df):,}")

## 2. Γεωγραφικά όρια

Κρατιούνται τα σημεία με πλάτος 40°–45° και μήκος 22,83°–25,63°. Ο τελικός περιορισμός στη Θεσσαλονίκη γίνεται στο βήμα 9.

In [ ]:
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 40.0, 45.0, 22.83, 25.63

n0 = len(df)
df = reindex(df[df.lat.between(LAT_MIN, LAT_MAX) & df.lon.between(LON_MIN, LON_MAX)].copy())
df = forward_diffs(df)
counts["Γεωγραφικά όρια"] = len(df)
print(f"εκτός ορίων: {n0 - len(df):,} | σημεία: {len(df):,}")

## 3. Στιγμιαία διπλότυπα

Αφαιρούνται τα σημεία με Δt = 0 (εκτός από το τελευταίο του αρχείου). Οι μετατοπίσεις επαναϋπολογίζονται στο βήμα 5.

In [ ]:
n0 = len(df)
df = reindex(df[(df.dt != 0.0) | (df.index == len(df) - 1)].copy())
counts["Στιγμιαία διπλότυπα (Δt = 0)"] = len(df)
print(f"Δt = 0: {n0 - len(df)} | σημεία: {len(df):,}")

## 4. Κεραίες

Υποψήφια κεραία = μπλοκ ≥ 2 διαδοχικών σημείων με ίδιες συντεταγμένες, που απέχει > 50 m από το προηγούμενο και από το επόμενο σημείο. Μετά από οπτικό έλεγχο:
- εξαιρούνται 9 μπλοκ που είναι στάσεις σε φανάρι,
- προστίθενται 5 μπλοκ που δεν πληρούν το κριτήριο .


In [ ]:
ANTENNA_DIST_M = 50.0
TRAFFIC_LIGHTS = [(192794, 192795), (187478, 187479), (169842, 169843), (129402, 129403), (83329, 83330),
                  (51281, 51282), (33528, 33530), (11965, 11966), (1700, 1701)]
EXTRA_ANTENNAS = [(182281, 182281), (72610, 72610), (72611, 72612), (133505, 133507), (5320, 5321), (35032, 35035)]

lat, lon, x, y, n = df.lat.to_numpy(), df.lon.to_numpy(), df.x.to_numpy(), df.y.to_numpy(), len(df)
candidates, i = [], 0
while i < n:
    j = i + 1
    while j < n and lat[j] == lat[i] and lon[j] == lon[i]:
        j += 1
    if j - i >= 2:
        d_in = np.hypot(x[i] - x[i - 1], y[i] - y[i - 1]) if i > 0 else 0.0
        d_out = np.hypot(x[j] - x[i], y[j] - y[i]) if j < n else 0.0
        if d_in > ANTENNA_DIST_M and d_out > ANTENNA_DIST_M:
            candidates.append((i, j - 1))
    i = j

to_delete = {b for b in candidates if b not in TRAFFIC_LIGHTS} | set(EXTRA_ANTENNAS)
drop = sorted({k for s, e in to_delete for k in range(s, e + 1)})
df = reindex(df.drop(index=drop))
counts["Κεραίες"] = len(df)
print(f"υποψήφια μπλοκ: {len(candidates)} | φανάρια: {sum(b in TRAFFIC_LIGHTS for b in candidates)} | "
      f"μπλοκ που αφαιρέθηκαν: {len(to_delete)} με {len(drop)} σημεία | σημεία: {len(df):,}")

## 5. Μετατοπίσεις

Επαναϋπολογισμός των `dt`, `dx`, `dy` και `t_j` μετά τις διαγραφές.

In [ ]:
df = forward_diffs(df)

## 6. Σημαίες κατάστασης B

- B = 0 (παύση): μηδενική μετατόπιση, ή B = 0 από το βήμα 1 με Δt ≤ 2 λεπτά και απόσταση < 50 m (τότε `dx = dy = 0`).
- B = 1 (πτήση): τα υπόλοιπα σημεία με Δt ≤ 2 λεπτά.
- B = 2 (κενό): 2 λεπτά < Δt ≤ 24 ώρες.
- B = 6 (διακοπή): Δt > 24 ώρες.

In [ ]:
DT_GAP_H, DT_BREAK_H, PAUSE_M = 2.0 / 60.0, 24.0, 50.0

dist = np.hypot(df.dx, df.dy)
df.loc[(df.dx == 0) & (df.dy == 0) & (df.B != 0), "B"] = 0
short_b0 = (df.B == 0) & (df.dt <= DT_GAP_H)
df.loc[short_b0 & (dist < PAUSE_M), ["B", "dx", "dy"]] = [0, 0.0, 0.0]
df.loc[short_b0 & (dist >= PAUSE_M), "B"] = 1
df.loc[(df.dt > DT_GAP_H) & (df.dt <= DT_BREAK_H), "B"] = 2
df.loc[df.dt > DT_BREAK_H, "B"] = 6
print("B:", df.B.value_counts().sort_index().to_dict())

## 7. Ταχύτητα

`speed_kmh` από τη μετατόπιση και το Δt.

In [ ]:
df["speed_kmh"] = np.where(df.dt > 0, np.hypot(df.dx, df.dy) / 1000.0 / df.dt, 0.0).round(2)
df["icon"] = np.where(df.B.isin([2, 6]), "lost", np.where(df.speed_kmh > 7.5, "car", "profile"))

## 8. Αλλαγή θερινής ώρας
Μετά τη διαγραφή επαναϋπολογίζονται τα `dt`, `dx`, `dy` σε όλο το αρχείο

In [ ]:
df = csv_roundtrip(df)
t = pd.to_datetime(df.real_time).to_numpy()
drop = []
for k in np.where(t[1:] < t[:-1])[0]:            # το ρολόι πήγε πίσω μετά το σημείο k
    j = k + 1
    while j < len(t) and t[j] <= t[k]:
        drop.append(j); j += 1
    print(f"επανάληψη ώρας: {df.real_time[k + 1]} → {df.real_time[j - 1]} ({j - k - 1} σημεία)")
df = reindex(df.drop(index=drop))

t = pd.to_datetime(df.real_time)
df["dt"] = (t.shift(-1) - t).dt.total_seconds() / 3600.0
df["dx"] = df.x.shift(-1) - df.x
df["dy"] = df.y.shift(-1) - df.y
assert (df.dt.iloc[:-1] > 0).all(), "ο χρόνος δεν είναι αυστηρά αύξων"
counts["Αλλαγή θερινής ώρας"] = len(df)
print(f"σημεία: {len(df):,}")

## 9. Διαδρομές εκτός Θεσσαλονίκης

Διαδρομή εκτός πόλης = συνεχόμενο μπλοκ σημείων σε > 12 km από την κατοικία, που φτάνει πάνω από 25 km. Το μπλοκ αφαιρείται. Το σημείο αμέσως πριν παίρνει B = 6, `dx = dy = 0` και `dt` ως το πρώτο σημείο μετά το μπλοκ.

In [ ]:
df = csv_roundtrip(df)
HOME_LAT =  None  # set to your own
HOME_LON= None  # set to your own
NEAR_KM, FAR_KM = 12.0, 25.0

dlat = np.radians(df.lat - HOME_LAT)
dlon = np.radians(df.lon - HOME_LON) * np.cos(np.radians(HOME_LAT))
dist_km = 6371.0 * np.sqrt(dlat ** 2 + dlon ** 2)

far, blocks, i = (dist_km > NEAR_KM).to_numpy(), [], 0
while i < len(df):
    if far[i]:
        j = i
        while j + 1 < len(df) and far[j + 1]:
            j += 1
        if dist_km[i:j + 1].max() > FAR_KM:
            blocks.append((i, j))
        i = j + 1
    else:
        i += 1

times = pd.to_datetime(df.real_time)
for s, e in blocks:
    p = s - 1
    df.loc[p, ["B", "dx", "dy"]] = [6, 0.0, 0.0]
    df.loc[p, "dt"] = round((times[e + 1] - times[p]).total_seconds() / 3600.0, 6)
    print(f"  {df.real_time[s]} → {df.real_time[e]} ({e - s + 1} σημεία)")
df = reindex(df.drop(index=[k for s, e in blocks for k in range(s, e + 1)]))
counts["Διαδρομές εκτός Θεσσαλονίκης"] = len(df)

df.to_csv(OUTPUT_CSV, index=False)
print(f"\n{OUTPUT_CSV}: {len(df):,} σημεία | B: {df.B.value_counts().sort_index().to_dict()}")
print(pd.Series(counts, name="σημεία").to_string())

## 10. Στατιστικά του τελικού συνόλου

- Κενά καταγραφής (B = 2) και ποσοστό τους έως 31 χαμένα σημεία, δηλαδή διάρκεια ≤ 32,5 λεπτά (Σχήμα 3.1).
- Τα 4 folds (Πίνακας 3.5): μέρες, σημεία και ώρες συνεχούς καταγραφής 

In [ ]:
gap_min = df.dt[df.B == 2] * 60
print(f"κενά B = 2: {len(gap_min):,} | έως 31 χαμένα σημεία (≤ 32,5 λεπτά): {(gap_min <= 32.5).mean():.0%}")

day = (pd.to_datetime(df.real_time) - pd.Timedelta(hours=5)).dt.strftime("%Y-%m-%d")
folds = np.array_split(np.sort(day.unique()), 4)
rows = []
for k, d in enumerate(folds, 1):
    m = day.isin(d)
    rows.append({"fold": k, "διάστημα": f"{pd.Timestamp(d[0]):%d/%m}–{pd.Timestamp(d[-1]):%d/%m}", "μέρες": len(d),
                 "σημεία": int(m.sum()), "ώρες": round(df.dt[m & df.B.isin([0, 1])].sum(), 1)})
print(pd.DataFrame(rows).to_string(index=False))